In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Read Inside Airbnb data").getOrCreate()

26/05/03 18:01:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
listings = spark.read.csv("listings.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")


In [3]:
listings.filter(listings.picture_url.isNotNull()).select('picture_url').limit(1).show(truncate=False)

+------------------------------------------------------------------------------------------------------+
|picture_url                                                                                           |
+------------------------------------------------------------------------------------------------------+
|https://a0.muscache.com/pictures/miso/Hosting-13913/original/d755aa6d-cebb-4464-80be-2722c921e8d5.jpeg|
+------------------------------------------------------------------------------------------------------+



In [4]:
# get number of properties that get more than 10 reviews per month
listings.filter(listings.reviews_per_month > 10).count()

66

In [6]:
# get properties that have more bathrooms than bedrooms
listings.filter((listings.bathrooms > listings.bedrooms)).select('name', 'bathrooms', 'bedrooms').show(truncate=False)

+-------------------------------------------------+---------+--------+
|name                                             |bathrooms|bedrooms|
+-------------------------------------------------+---------+--------+
|Cosy Double studio in Zone 2 Hammersmith (1)     |1.5      |1       |
|LONDON DETACHED HOUSE*ElecGates etc              |2.0      |1       |
|Designer room Park Views 4 mins zone 1 station   |1.5      |1       |
|Maisonette in Central London Zone 1              |1.5      |1       |
|West London,loft ensuite, 5min2tube              |1.5      |1       |
|Shoreditch Loft                                  |1.5      |1       |
|Five minute walk to South Bank                   |1.5      |1       |
|Stunning double room own bathroom                |4.0      |1       |
|Also five minutes to South Bank                  |1.5      |1       |
|Studio 20 min from center                        |1.0      |0       |
|Spacious luxury 2 bedroom apartment              |1.5      |1       |
|Singl

In [9]:
# get properties where the price is greater than 5.000, collect the result as a python list
from pyspark.sql.functions import regexp_replace

listings_with_price = listings.withColumn('price_numeric', regexp_replace('price', '[$,]', '').cast('float'))

res = listings_with_price.filter(listings_with_price.price_numeric > 5000).select('name', 'price').collect()
res

[Row(name='Room in a cosy flat. Central, clean', price='$8,000.00'),
 Row(name='Spacious Private Ground Floor Room', price='$6,309.00'),
 Row(name='No Longer Available', price='$53,588.00'),
 Row(name='Bright & airy DoubleBed with EnSuite in Zone 2!', price='$74,100.00'),
 Row(name='The Apartments by The Sloane Club, Two Bedroom Apt', price='$7,377.00'),
 Row(name='The Apartments by The Sloane Club, L 2 Bedroom Apt', price='$7,377.00'),
 Row(name='Single room. 7ft x 9ft - Over looking garden', price='$6,523.00'),
 Row(name='Close To London Eye (TUR)', price='$6,666.00'),
 Row(name='Beautiful 2 BR flat in Kilburn with free parking', price='$6,000.00'),
 Row(name='Semi-detached mews house in Knightsbridge.', price='$7,019.00'),
 Row(name='Affordable Spacious  Room on the edge of the city', price='$6,000.00'),
 Row(name='Henry’s Townhouse, London', price='$6,500.00'),
 Row(name='City Suite', price='$5,353.00'),
 Row(name='Hyde Park Suite', price='$5,653.00'),
 Row(name='SHORT WALK TO LOND

In [12]:
# get a list of properties with the following characteristics:
# price < 150
# more than 20 reviews
# review_scores_rating > 4.5
# consider using the "&" operator

listings_with_price.filter(
    (listings_with_price.price_numeric < 150) &
    (listings_with_price.number_of_reviews > 20) &
    (listings_with_price.review_scores_rating > 4.5)
).select('name', 'price_numeric', 'number_of_reviews', 'review_scores_rating').show(truncate=False)

+-------------------------------------------------+-------------+-----------------+--------------------+
|name                                             |price_numeric|number_of_reviews|review_scores_rating|
+-------------------------------------------------+-------------+-----------------+--------------------+
|Holiday London DB Room Let-on going              |70.0         |55               |4.85                |
|Bright Chelsea  Apartment. Chelsea!              |149.0        |97               |4.8                 |
|You are GUARANTEED to love this                  |90.0         |730              |4.87                |
|SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST       |61.0         |387              |4.77                |
|SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT         |49.0         |72               |4.97                |
|Room with a view, shared flat,  central  Bankside|96.0         |137              |4.7                 |
|You Will Save Money Here                         |71.0

In [13]:
# get a list of properties with the following characteristics:
# price < 150 or more than one bathroom
# use the "|" operator to implement the OR operator
listings_with_price.filter((listings_with_price.price_numeric < 150) | (listings_with_price.bedrooms > 1)).select('name', 'price_numeric', 'bedrooms').show(truncate=False)

+--------------------------------------------------+-------------+--------+
|name                                              |price_numeric|bedrooms|
+--------------------------------------------------+-------------+--------+
|Holiday London DB Room Let-on going               |70.0         |1       |
|Bright Chelsea  Apartment. Chelsea!               |149.0        |1       |
|Very Central Modern 3-Bed/2 Bath By Oxford St W1  |411.0        |3       |
|Kew Gardens 3BR house in cul-de-sac               |280.0        |3       |
|You are GUARANTEED to love this                   |90.0         |1       |
|SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST        |61.0         |1       |
|Short Term Home                                   |340.0        |4       |
|SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT          |49.0         |1       |
|2 Double bed apartment in quiet area North London |213.0        |2       |
|Room with a view, shared flat,  central  Bankside |96.0         |1       |
|Room in rel

In [15]:
# get the highest listings price in this dataset consider using the "max" function from "pyspark.sql.functions"
from pyspark.sql.functions import max
listings_with_price.select(max('price_numeric')).show()

[Stage 11:>                                                         (0 + 1) / 1]

+------------------+
|max(price_numeric)|
+------------------+
|         1085147.0|
+------------------+



In [18]:
# get the name and a price of property with the highest number of reviews per month try to use "collect"
# method to get the price first, and then use it in a "filter" call
res = listings_with_price.select(max('price_numeric').alias('max_price')).collect()
res
max_price = res[0]['max_price']
max_price
listings_with_price.filter(listings_with_price.price_numeric == max_price).select('name', 'price').show()

[Stage 23:>                                                         (0 + 1) / 1]

+--------------------+-------------+
|                name|        price|
+--------------------+-------------+
|Lux 2 Bed in Cana...|$1,085,147.00|
+--------------------+-------------+



In [19]:
# get the number of hosts in the dataset
listings.select('host_name').distinct().count()

16673

In [21]:
# get listings with a first review in 2024 consider using the "year" function from "pyspark.sql.functions"
from pyspark.sql.functions import year

listings.filter(year(listings.first_review) == 2024).select('name', 'first_review').show(10, truncate=False)

+--------------------------------------------------+------------+
|name                                              |first_review|
+--------------------------------------------------+------------+
|Close to Wimbledon All England Tennis -huge double|2024-08-11  |
|one Double bed room with en-suite facilities      |2024-03-21  |
|Bridgerton inspired cottage core apartment        |2024-09-14  |
|Sm double room  with own bathroom                 |2024-06-04  |
|Central, modern pied-a-terre                      |2024-11-29  |
|Superlux flat in Knightsbridge                    |2024-01-01  |
|The Pink House, Notting Hill                      |2024-07-14  |
|Stylish garden flat in Hackney                    |2024-09-15  |
|Luxurious Flat in South Kensington                |2024-06-19  |
|Double Standard Room (Ensuite)                    |2024-09-01  |
+--------------------------------------------------+------------+
only showing top 10 rows
